# 🚦 Assistant Juridique — Code de la Route Marocain

> **Pipeline NLP complet** : Chargement CSV → Clustering TF-IDF → Embeddings multilingues → Index FAISS → RAG (Qwen) → Interface Gradio  
> Le fichier `export_final.csv` est déjà disponible ; les cellules d'extraction PDF sont incluses à titre optionnel.

In [1]:
# ─────────────────────────────────────────────
# INSTALLATION DES DÉPENDANCES
# ─────────────────────────────────────────────
%pip install -q PyMuPDF pyarabic pandas scikit-learn \
               sentence-transformers faiss-cpu \
               langchain langchain-community langchain-text-splitters \
               transformers torch gradio

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\HP\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [2]:
import re, os, warnings
import numpy as np
import pandas as pd
import pyarabic.araby as araby
import torch
import faiss
import gradio as gr

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

warnings.filterwarnings("ignore")
print("✅ Tous les imports sont OK.")

c:\Users\HP\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Tous les imports sont OK.


## (Optionnel) Étape 0 — Extraction du texte brut depuis le PDF

Cette cellule n'est utile que si vous souhaitez **regénérer** le CSV depuis le document source.  
Si vous avez déjà `export_final.csv`, passez directement à l'étape 1.

In [3]:
CHEMIN_PDF = "/content/code de la route MA52_05.pdf"  # ← adaptez ce chemin

def extraire_texte_pdf(chemin):
    """Extrait le texte brut de toutes les pages d'un PDF via PyMuPDF."""
    import fitz
    doc = fitz.open(chemin)
    return "\n".join(page.get_text() for page in doc)

def nettoyer_texte(texte):
    """Nettoie un texte juridique arabe extrait d'un PDF."""
    texte = re.sub(r"(^|\s)امل(?=\w)", r"\1الم", texte)
    texte = texte.replace("إال", "إلا").replace("وال ", "ولا ")
    texte = re.sub(r"(\d+)\s+(المادة)", r"\2 \1", texte)
    texte = araby.strip_tashkeel(texte)
    texte = re.sub(r"[أإآ]", "ا", texte)
    texte = texte.replace("ة", "ه")
    return re.sub(r"\s+", " ", texte).strip()

if Path(CHEMIN_PDF).exists():
    texte_brut  = extraire_texte_pdf(CHEMIN_PDF)
    texte_propre = nettoyer_texte(texte_brut)
    print(f"📄 {len(texte_brut):,} caractères extraits et nettoyés.")
else:
    texte_propre = None
    print("⚠️  PDF introuvable — cellule ignorée.")
    print("   Chargez export_final.csv à l'étape suivante.")

⚠️  PDF introuvable — cellule ignorée.
   Chargez export_final.csv à l'étape suivante.


## Étape 1 — Chargement du CSV exporté

In [4]:
# ─────────────────────────────────────────────
# CHARGEMENT DU FICHIER EXPORT_FINAL.CSV
# ─────────────────────────────────────────────

CSV_CANDIDATES = [
    "export_final.csv",
    "/content/export_final.csv",
    "/mnt/user-data/uploads/export_final.csv",
]

csv_path = next((p for p in CSV_CANDIDATES if Path(p).exists()), None)

if csv_path is None:
    raise FileNotFoundError(
        "export_final.csv introuvable. "
        "Uploadez le fichier ou modifiez CSV_CANDIDATES."
    )

df = pd.read_csv(csv_path, encoding="utf-8-sig")
print(f"✅ CSV chargé depuis : {csv_path}")
print(f"   {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"   Colonnes : {df.columns.tolist()}")
print()
print(df.head(3).to_string(index=False))

✅ CSV chargé depuis : export_final.csv
   336 lignes × 9 colonnes
   Colonnes : ['article_id', 'infraction_desc', 'amende_fixe', 'points_retrait', 'categorie_vehicule', 'type_infraction', 'mots_cles', 'cluster', 'categorie_cluster']

 article_id                                                                                                                                                                                                                                            infraction_desc  amende_fixe  points_retrait categorie_vehicule type_infraction mots_cles  cluster      categorie_cluster
          1                                              1 املاده ال يجوز الي شخص ان يسوق مركبه ذات محرك او مجموعه مركبات علا الطريق العموميه ما لم يكن حاصال علا رخصه للسياقه ساريه الصالحيه ومسلمه من قبل االداره، تناسب صنف .املركبه او مجموعه املركبات التي يسوقها          NaN             NaN              leger      definition documents        3     regles_circulation
          2 2 املاده استثناء

## Étape 2 — Vectorisation TF-IDF et clustering K-Means

In [5]:
# ─────────────────────────────────────────────
# CLUSTERING TF-IDF
# ─────────────────────────────────────────────

N_CLUSTERS = 6

col_texte = "infraction_desc" if "infraction_desc" in df.columns else df.columns[1]
textes_bruts = df[col_texte].fillna("").astype(str)

vectorizer = TfidfVectorizer(max_features=200, min_df=1)
X_tfidf    = vectorizer.fit_transform(textes_bruts)

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
df["cluster_tfidf"] = kmeans.fit_predict(X_tfidf)

print("✅ Clustering TF-IDF terminé.")
print(df["cluster_tfidf"].value_counts().sort_index().to_string())

✅ Clustering TF-IDF terminé.
cluster_tfidf
0    78
1    53
2    77
3    33
4    44
5    51


## Étape 3 — Embeddings multilingues (Sentence-Transformers)

In [6]:
EMBEDDING_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"

print(f"⏳ Chargement du modèle : {EMBEDDING_MODEL} …")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

textes_list = textes_bruts.tolist()

print(f"⏳ Encodage de {len(textes_list)} documents …")
embeddings = embedding_model.encode(
    textes_list,
    show_progress_bar=True,
    batch_size=32,
)

print(f"✅ Matrice d'embeddings : {embeddings.shape}")

⏳ Chargement du modèle : paraphrase-multilingual-MiniLM-L12-v2 …


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6663.79it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⏳ Encodage de 336 documents …


Batches: 100%|██████████| 11/11 [00:11<00:00,  1.08s/it]

✅ Matrice d'embeddings : (336, 384)


## Étape 4 — Index FAISS et fonction Retriever

In [7]:
# ─────────────────────────────────────────────
# CREATION DE L'INDEX FAISS
# ─────────────────────────────────────────────

dimension = embeddings.shape[1]
index     = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype("float32"))

print(f"✅ Index FAISS : {index.ntotal} vecteurs (dim={dimension})")


def retrieve(query: str, k: int = 4) -> list:
    """Recherche les k documents les plus proches dans l'index FAISS."""
    query_vec = embedding_model.encode([query]).astype("float32")
    distances, indices = index.search(query_vec, k)
    return [textes_list[i] for i in indices[0] if i < len(textes_list)]


# ── Test rapide ──────────────────────────────────────────────
q_test = "ما هي غرامة التوقف غير القانوني؟"
print(f"\nTest FAISS : \"{q_test}\"\n")
for i, doc in enumerate(retrieve(q_test, k=3), 1):
    print(f"Résultat {i}: {doc[:200]}\n")

✅ Index FAISS : 336 vecteurs (dim=384)

Test FAISS : "ما هي غرامة التوقف غير القانوني؟"

Résultat 1: 108 املاده يرفع التوقيف، ما لم توجد احكام مخالفه في عين املكان، من قبل العون محرر املحضر، الذي امر به وذلك فور انهاء املخالفه ؛ 1 من قبل السلطه املؤهله التابع لها العون محرر املحضر واملرفوع اليها االم

Résultat 2: 107 املاده اذا لم يتم انهاء املخالفه التي بررت التوقيف، وقت مغادره العون محرر املحضر ملكان ايقاف املركبه، يقوم هذا العون برفع االمر الا االداره التابع لها مع تسليمها شهاده التسجيل او رخصه .السياقه حسب

Résultat 3: 98 املاده 96 و95 يتوقف اثر توقيف او سحب رخصه السياقه املقرر من قبل االداره وفقا الحكام املواد ، اعاله، كيفما كانت مدته، بعد اي قرار بالحفظ تصدره النيابه العامه او اذا اصبح قابال للتنفيذ97و من اجل نفس 



## Étape 5 — Construction des chunks LangChain

In [8]:
def build_documents(df):
    """Construit des Documents LangChain enrichis depuis le DataFrame."""
    docs = []
    for _, row in df.iterrows():
        content = f"حسب المادة {row['article_id']} من مدونة السير: "
        if pd.notna(row.get("infraction_desc", None)):
            content += f"{row['infraction_desc']}. "
        if pd.notna(row.get("amende_fixe", None)):
            content += f"الغرامة: {row['amende_fixe']} درهم. "
        if pd.notna(row.get("points_retrait", None)):
            content += f"النقط المخصومة: {row['points_retrait']}. "
        cat = row.get("categorie_vehicule", "")
        if pd.notna(cat) and cat not in ("Non spécifié", "leger", ""):
            content += f"صنف المركبة: {cat}. "
        docs.append(Document(
            page_content=content,
            metadata={
                "article_id": str(row["article_id"]),
                "mots_cles":  str(row.get("mots_cles", "")),
            }
        ))
    return docs


splitter  = RecursiveCharacterTextSplitter(chunk_size=350, chunk_overlap=50)
documents = build_documents(df)
chunks    = splitter.split_documents(documents)

print(f"✅ {len(documents)} documents → {len(chunks)} chunks")
print("\nExemple de chunk :")
print(chunks[0].page_content)

✅ 336 documents → 336 chunks

Exemple de chunk :
حسب المادة 1 من مدونة السير: 1 املاده ال يجوز الي شخص ان يسوق مركبه ذات محرك او مجموعه مركبات علا الطريق العموميه ما لم يكن حاصال علا رخصه للسياقه ساريه الصالحيه ومسلمه من قبل االداره، تناسب صنف .املركبه او مجموعه املركبات التي يسوقها.


In [27]:
%pip install bitsandbytes==0.43.1

   ---------------------------------------- 0.0/101.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.6 MB ? eta -:--:--
   ---------------------------------------- 0.8/101.6 MB 1.2 MB/s eta 0:01:22
   ---------------------------------------- 1.0/101.6 MB 1.4 MB/s eta 0:01:12
    --------------------------------------- 1.6/101.6 MB 1.7 MB/s eta 0:01:00
    --------------------------------------- 2.4/101.6 MB 2.0 MB/s eta 0:00:51
   - -------------------------------------- 3.1/101.6 MB 2.3 MB/s eta 0:00:44
   - -------------------------------------- 4.5/101.6 MB 2.8 MB/s eta 0:00:36
   -- ------------------------------------- 6.0/101.6 MB 3.3 MB/s eta 0:00:29
   --- ------------------------------------ 7.9/101.6 MB 3.9 MB/s eta 0:00:24
   ---- ----------------------------------- 10.5/101.6 MB 4.7 MB/s eta 0:00:20
   ---- -----

## Étape 6 — RAG avec Qwen2.5-0.5B-Instruct

In [29]:
LLM_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"


print(f"⏳ Chargement du LLM : {LLM_MODEL} …")
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    # device_map="auto",   ← remove this line
    torch_dtype=torch.float32,  # float16 only useful on GPU
)

generator = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=llm_tokenizer,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.3,
    return_full_text=False,
)
print("✅ LLM chargé.")


def rag_answer(query: str, k: int = 4) -> str:
    """Pipeline RAG : Retrieve → Augment → Generate."""
    docs    = retrieve(query, k=k)
    context = "\n".join(docs)
    prompt  = (
        "أنت خبير في قوانين مدونة السير المغربية.\n"
        "استخدم فقط السياق التالي للإجابة بدقة على السؤال.\n\n"
        f"السياق:\n{context}\n\n"
        f"السؤال:\n{query}\n\n"
        "الجواب:"
    )
    out = generator(prompt)
    return out[0]["generated_text"].strip()


# ── Test ─────────────────────────────────────────────────────
question = "ما هي غرامة التوقف غير القانوني؟"
print(f"Question : {question}\n")
print("⏳ Génération de la réponse …")
reponse = rag_answer(question)
print("=" * 60)
print(f"Réponse :\n{reponse}")
print("=" * 60)

⏳ Chargement du LLM : Qwen/Qwen2.5-0.5B-Instruct …


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 476.26it/s]
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ LLM chargé.
Question : ما هي غرامة التوقف غير القانوني؟

⏳ Génération de la réponse …
Réponse :
1. 108 املاده يرفع التوقيف، ما لم توجد احكام مخالفه في عين املكان، من قبل العون محرر املحضر، الذي امر به وذلك فور انهاء املخالفه ؛
2. 1 من قبل السلطه املؤهله التابع لها العون محرر املحضر واملرفوع اليها االمر وفق الشروط 2 ، اعاله، بمجرد ما يثبت السائق انتها
3. 107 املاده إذا لم يتم انهاء املخالفه التي بررت التوقيف، وقت مغادره العون محرر املحضر ملكان أيقاف املركبه، يقوم هذا العون برفع االمر الا االداره التابع لها مع تسليمها شهاده التسجيل او ر


In [34]:
%pip uninstall gradio -y
%pip uninstall gradio-client -y
%pip install gradio==4.44.1

Found existing installation: gradio 3.50.2
Uninstalling gradio-3.50.2:
  Successfully uninstalled gradio-3.50.2
Note: you may need to restart the kernel to use updated packages.
Found existing installation: gradio_client 0.6.1
Uninstalling gradio_client-0.6.1:
  Successfully uninstalled gradio_client-0.6.1
Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/18.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.1 MB ? eta -:--:--
    --------------------------------------- 0.3/18.1 MB ? eta -:--:--
   - -------------------------------------- 0.5/18.1 MB 1.1 MB/s eta 0:00:16
   - -------------------------------------- 0.8/18.1 MB 1.2 MB/s eta 0:00:14
   -- ------------------------------------- 1.3/18.1 MB 1.5 MB/s eta 0:00:12
   ---- ----------------------------------- 1.8/18.1 MB 1.7 MB/s eta 0:00:10
   ----- ---------------------------------- 2.6/18.1 MB 2.0 MB/s eta 0:00:08
   ------- --------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.3.0 which is incompatible.


## Étape 7 — Interface Gradio

In [38]:
docs = retriever.get_relevant_documents(question)

for i, doc in enumerate(docs):
    print(f"\n--- DOC {i} ---\n", doc.page_content)
    
def interface_rag(question):
    try:
        if not question.strip():
            return "⚠️ Veuillez poser une question."
        return rag_answer(question)
    except Exception as e:
        return f"❌ Erreur interne: {str(e)}"

demo = gr.Interface(
    fn=interface_rag,
    inputs=gr.Textbox(
        lines=3,
        placeholder="اطرح سؤالك حول مدونة السير هنا…",
        label="❓ سؤالك / Votre question :",
    ),
    outputs=gr.Textbox(label="🤖 Réponse de l'assistant :", lines=6),
    title="🚦 Assistant Intelligent — Code de la Route Marocain",
    description="RAG + FAISS + Qwen",
    examples=[
        ["ما هي غرامة التوقف غير القانوني؟"],
        ["كم عدد النقط التي تُخصم عند تجاوز السرعة؟"],
        ["ما هي عقوبة قيادة دراجة نارية بدون خوذة؟"],
    ],
    theme=gr.themes.Soft(),
    flagging_mode="never",
)

print("Question:", question)
print(rag_answer("ما هي عقوبة تجاوز السرعة؟"))

print("🚀 Lancement de l'interface Gradio …")
demo.launch(share=True, debug=True)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: ما هي غرامة التوقف غير القانوني؟
150 املاده من القانون الجنائي ودون خالله بالعقاب 361 في معايبه امل نصوص عليها في الفصل الا5.000 اشهر وبغرامه من خمسه الف درهم 6 االشد، يعاقب بالحبس من شهر واحد الا سته درهم او باحدا هاتين العقوبتين فقط، كل شخص حصل بعد اجتياز امتحان 20.000 عشر

الموضوع الذي يخص هذه القاعدة هو:

القانون الجنائي والحكم عليه بالحبس من شهر واحد الا سته درهم او باحدا هاتين العقوبتين فقط، كل شخص حصل بعد اجتياز امتحان 20.000 عشره الف 2.000 اشهر وبغرامه من الفين كل سائق ا
🚀 Lancement de l'interface Gradio …
* Running on local URL:  http://127.0.0.1:7861

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/04/21 18:14:59 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: i/o timeout


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> None
